# 08 — End-to-End: Real LLM Driving MCP Tools

**What you'll learn**
- The shape of the agentic loop: **prompt → tool_use → tool_result → continue**.
- What the LLM actually receives — the JSON Schema generated from your tools.
- How to wire Anthropic's Claude API into the loop and read `stop_reason="tool_use"`.
- A mock LLM that ships with the notebook so every cell runs without an API key, and a real Claude path that turns on with `ANTHROPIC_API_KEY`.

## The loop

```text
+--------+    messages       +--------+
| Host   | ---------------->|  LLM   |
|        | (incl. tool defs) |        |
|        | <----------------|        |
|        |  tool_use         +--------+
|        |
|        |  tools/call       +--------+
|        | ---------------->| MCP    |
|        |                   | server |
|        | <----------------|        |
|        |  result           +--------+
|        |
|        |  messages         +--------+
|        | ---------------->|  LLM   |
|        | (incl. result)    |        |
|        | <----------------|        |
|        |  text answer      +--------+
+--------+
```

The host keeps looping: every time the model says `tool_use`, the host runs the tool through MCP and feeds the result back. The loop ends when the model returns plain text (`stop_reason == "end_turn"`).

## A small MCP server with explicit schemas

Up to now we let `MiniMCPServer` infer descriptions from docstrings. For the LLM to actually pick the right tool, it needs a JSON Schema for each argument. Real FastMCP generates this from your type hints. Here we write the schema by hand so you can see exactly what the model receives.

In [ ]:
from typing import Callable, Any

class MiniMCPServer:
    def __init__(self, name: str) -> None:
        self.name = name
        self._tools: dict[str, Callable[..., Any]] = {}
        self._meta: dict[str, dict] = {}

    def tool(self, description: str, input_schema: dict):
        """Register a tool with an explicit JSON Schema — same shape the LLM sees."""
        def deco(fn):
            self._tools[fn.__name__] = fn
            self._meta[fn.__name__] = {"description": description,
                                        "input_schema": input_schema}
            return fn
        return deco

    def list_tools(self) -> list[dict]:
        """Return tool definitions in the shape Anthropic's API expects."""
        return [{"name": n,
                 "description": self._meta[n]["description"],
                 "input_schema": self._meta[n]["input_schema"]}
                for n in self._tools]

    def call_tool(self, name: str, arguments: dict):
        return self._tools[name](**arguments)

In [ ]:
CONTACTS: dict = {}

server = MiniMCPServer("sales-tools-llm")


@server.tool(
    description="Create a new contact in the CRM.",
    input_schema={
        "type": "object",
        "properties": {
            "name":  {"type": "string", "description": "Full name"},
            "email": {"type": "string", "format": "email"},
        },
        "required": ["name", "email"],
    },
)
def create_contact(name: str, email: str) -> dict:
    cid = f"contact_{len(CONTACTS) + 1}"
    CONTACTS[cid] = {"id": cid, "name": name, "email": email}
    return CONTACTS[cid]


@server.tool(
    description="Look up a contact by id.",
    input_schema={
        "type": "object",
        "properties": {"contact_id": {"type": "string"}},
        "required": ["contact_id"],
    },
)
def get_contact(contact_id: str) -> dict:
    if contact_id not in CONTACTS:
        return {"error": f"no such contact: {contact_id}"}
    return CONTACTS[contact_id]


import json
print(json.dumps(server.list_tools(), indent=2))

**That JSON above is exactly what the LLM sees.** Internalize the shape:

```json
{
  "name": "create_contact",
  "description": "...",
  "input_schema": {
    "type": "object",
    "properties": {...},
    "required": [...]
  }
}
```

Same shape across OpenAI, Anthropic, Gemini (with tiny key-name differences). When you write a tool, you're really writing one of these schemas.

## The agent loop

Pseudocode you can run regardless of which LLM provider you use:

```text
messages = [user_message]
loop:
    response = llm.create(model=..., messages=messages, tools=server.list_tools())
    messages.append(assistant_response)
    if response.stop_reason == "end_turn":
        return last text block
    for tool_use_block in response.content:
        result = server.call_tool(tool_use_block.name, tool_use_block.input)
        messages.append(user_with_tool_result(tool_use_block.id, result))
```

The whole loop in eight lines. Below we implement it with a swappable `llm_client`.

In [ ]:
def run_agent(llm_client, user_message: str, max_turns: int = 6) -> dict:
    messages = [{"role": "user", "content": user_message}]
    transcript = []
    for turn in range(max_turns):
        response = llm_client.create(
            messages=messages,
            tools=server.list_tools(),
        )
        transcript.append({"turn": turn, "response": response})

        # Add the assistant turn to the message history.
        messages.append({"role": "assistant", "content": response["content"]})

        if response["stop_reason"] == "end_turn":
            text = next((b["text"] for b in response["content"] if b["type"] == "text"), "")
            return {"answer": text, "transcript": transcript, "messages": messages}

        if response["stop_reason"] == "tool_use":
            tool_results = []
            for block in response["content"]:
                if block["type"] != "tool_use":
                    continue
                result = server.call_tool(block["name"], block["input"])
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block["id"],
                    "content": json.dumps(result),
                })
            # The LLM expects tool results as a single user message
            # containing one tool_result block per tool_use.
            messages.append({"role": "user", "content": tool_results})
            continue

        raise RuntimeError(f"unexpected stop_reason: {response['stop_reason']}")

    raise RuntimeError("exceeded max_turns without end_turn")

## Mock LLM — runs offline

Returns responses in the exact same shape that `anthropic.Anthropic().messages.create()` returns (after a small `.model_dump()` style conversion to plain dicts). The agent loop above doesn't know it's a mock.

In [ ]:
import itertools

class MockClaude:
    """Deterministic LLM stand-in. Walks through a scripted plan."""

    def __init__(self) -> None:
        self._ids = itertools.count(1)

    def _tu_id(self) -> str:
        return f"tu_{next(self._ids):03d}"

    def create(self, messages, tools):
        # Use the last message to decide what to do next.
        last = messages[-1]
        if last["role"] == "user" and isinstance(last["content"], str):
            # First call: plan a tool_use.
            return {
                "stop_reason": "tool_use",
                "content": [
                    {"type": "text",
                     "text": "I will create the contact, then look it up to confirm."},
                    {"type": "tool_use", "id": self._tu_id(),
                     "name": "create_contact",
                     "input": {"name": "Grace Hopper",
                               "email": "grace@example.com"}},
                ],
            }

        if last["role"] == "user" and isinstance(last["content"], list):
            # Tool results came back. Inspect them.
            last_result = json.loads(last["content"][-1]["content"])
            if "id" in last_result and last["content"][-1]["tool_use_id"].startswith("tu_001"):
                # Got the created contact id — call get_contact to confirm.
                return {
                    "stop_reason": "tool_use",
                    "content": [
                        {"type": "tool_use", "id": self._tu_id(),
                         "name": "get_contact",
                         "input": {"contact_id": last_result["id"]}},
                    ],
                }
            # Second tool result — wrap up.
            return {
                "stop_reason": "end_turn",
                "content": [
                    {"type": "text",
                     "text": f"Created and verified contact: {last_result}"},
                ],
            }

        raise RuntimeError("MockClaude got an unexpected state")

## Run it end-to-end (offline)

In [ ]:
mock = MockClaude()
out = run_agent(mock, "Please create a new contact for Grace Hopper at grace@example.com, then confirm it.")
print("ANSWER:", out["answer"])
print()
print("CONTACTS:", CONTACTS)
print()
print(f"Loop ran {len(out['transcript'])} turn(s).")

### Trace through what happened

1. **Turn 0** — model received the user message + tool schemas, decided to call `create_contact(name="Grace Hopper", email="grace@example.com")`.
2. The host executed that via `server.call_tool` and got back `{"id": "contact_1", ...}`.
3. **Turn 1** — model saw the tool result, decided to call `get_contact("contact_1")` to confirm.
4. The host executed and returned the contact dict.
5. **Turn 2** — model had everything it needed, returned `end_turn` with a plain-text summary.

That two-tool-call dance is the standard "verify after you write" pattern. The model decides; the host executes; the MCP server provides the capabilities.

## Mini test

In [ ]:
assert out["answer"].startswith("Created and verified")
assert any(b.get("name") == "create_contact"
           for t in out["transcript"]
           for b in t["response"]["content"]
           if b.get("type") == "tool_use")
assert any(b.get("name") == "get_contact"
           for t in out["transcript"]
           for b in t["response"]["content"]
           if b.get("type") == "tool_use")
assert "contact_1" in CONTACTS
print("ok")

## The real thing — Anthropic SDK

Swap `MockClaude` for a thin adapter over `anthropic.Anthropic`. The same `run_agent` works unchanged. **You need `pip install anthropic` and `ANTHROPIC_API_KEY` set.**

In [ ]:
REAL_ANTHROPIC_CLIENT = r'''# anthropic_client.py
import os
from anthropic import Anthropic

class RealClaude:
    """Adapter so `run_agent` can talk to the real Anthropic API."""

    def __init__(self, model: str = "claude-opus-4-7"):
        self.model = model
        self.client = Anthropic()  # reads ANTHROPIC_API_KEY

    def create(self, messages, tools):
        resp = self.client.messages.create(
            model=self.model,
            max_tokens=1024,
            tools=tools,
            messages=messages,
        )
        # Convert the SDK objects to the plain-dict shape `run_agent` expects.
        return {
            "stop_reason": resp.stop_reason,
            "content": [
                ({"type": "text", "text": b.text} if b.type == "text"
                 else {"type": "tool_use", "id": b.id, "name": b.name, "input": b.input})
                for b in resp.content
            ],
        }
'''
print(REAL_ANTHROPIC_CLIENT)

# To run for real, in a notebook cell:
#   %pip install --quiet anthropic
#   import os; os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."
#   exec(REAL_ANTHROPIC_CLIENT)
#   out = run_agent(RealClaude(), "Create a contact for Grace Hopper at grace@example.com and confirm.")
#   print(out["answer"])

## Optional: live run, if you have a key

This cell only does something real if `ANTHROPIC_API_KEY` is set **and** the `anthropic` package is installed. Otherwise it skips silently.

In [ ]:
import os, importlib.util

if os.getenv("ANTHROPIC_API_KEY") and importlib.util.find_spec("anthropic") is not None:
    # Reset state for a clean run
    CONTACTS.clear()
    exec(REAL_ANTHROPIC_CLIENT)  # defines RealClaude
    real_client = RealClaude(model=os.getenv("ANTHROPIC_MODEL", "claude-opus-4-7"))
    out = run_agent(real_client,
                    "Create a contact for Grace Hopper at grace@example.com, then confirm it exists.")
    print("LIVE ANSWER:", out["answer"])
    print("CONTACTS:", CONTACTS)
else:
    print("skipped — set ANTHROPIC_API_KEY and pip install anthropic to run live")

## Key takeaway

The agent loop is **eight lines of code** plus a tool schema. What MCP buys you is the second half of those eight lines — `server.call_tool` is your only integration point, and every tool you ever add slots in behind it. Switch hosts (Claude Desktop, your own agent, a cron job), switch models, switch transports — the loop doesn't change. That's the whole point of the protocol.